# Lightweight Dual-Camera Explainable Anomaly Detection
## Google Colab Setup & Pipeline

**Before running:**
1. Upload `mvtec_3d_anomaly_detection.tar.xz` to the Colab files panel (left sidebar) or to Google Drive
2. Make sure GPU is enabled: Runtime → Change runtime type → T4 GPU

In [ ]:
# Check GPU
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU detected. Go to Runtime → Change runtime type → GPU")

## Step 1: Install dependencies

In [ ]:
!pip install -q timm scikit-learn opencv-python-headless pyyaml tifffile transformers

## Step 2: Clone or upload project
Choose ONE of the options below.

In [ ]:
# OPTION A: If project is on GitHub, uncomment and run:
# !git clone https://github.com/YOUR_USERNAME/YOUR_REPO.git
# %cd YOUR_REPO

# OPTION B: Upload project as zip to Colab files panel, then run:
# !unzip -q /content/YOUR_PROJECT.zip -d /content/
# %cd YOUR_PROJECT_FOLDER

# OPTION C: Mount Google Drive and navigate to project:
# from google.colab import drive
# drive.mount('/content/drive')
# %cd /content/drive/MyDrive/YOUR_PROJECT_FOLDER

# Verify we're in the right place (should see configs/, src/, etc.)
!ls

## Step 3: Extract dataset
Upload `mvtec_3d_anomaly_detection.tar.xz` to the Colab files panel, then run this.

In [ ]:
import os

# Find the uploaded tar file
tar_path = None
for f in os.listdir('.'):
    if f.endswith('.tar.xz'):
        tar_path = f
        break
if tar_path is None:
    # Check /content/ for uploaded files
    for f in os.listdir('/content/'):
        if f.endswith('.tar.xz'):
            tar_path = f'/content/{f}'
            break

if tar_path:
    print(f"Found dataset: {tar_path}")
    os.makedirs('data', exist_ok=True)
    !tar -xf {tar_path} -C data --strip-components=1
    print("Extracted. Categories:")
    !ls data/
else:
    print("No .tar.xz found. Upload mvtec_3d_anomaly_detection.tar.xz to the files panel.")

## Step 4: Extract features (with CLIP text tokens)
This runs RGB backbone, depth backbone, and CLIP on every image. First run downloads CLIP (~600MB).

In [ ]:
!python -m src.scripts.extract_features

## Step 5: Train fusion model (all categories)

In [ ]:
import yaml

with open('configs/config.yaml') as f:
    config = yaml.safe_load(f)

for cat in config['dataset']['categories']:
    print(f"\n{'='*50}")
    print(f"Training: {cat}")
    print(f"{'='*50}")
    !python -m src.scripts.train_fusion --category {cat} --epochs 10

## Step 6: Evaluate all categories

In [ ]:
import yaml

with open('configs/config.yaml') as f:
    config = yaml.safe_load(f)

for cat in config['dataset']['categories']:
    !python -m src.scripts.evaluate --category {cat}

## Step 7: Generate results report

In [ ]:
!python -m src.scripts.generate_report

## Step 8: Download outputs (checkpoints, features)
Zip and download the trained models and cached features.

In [ ]:
!tar -cf outputs.tar outputs/
from google.colab import files
files.download('outputs.tar')

## Quick Sanity Check
Run one category end-to-end to verify GPU is working before doing all 4.

In [ ]:
import torch
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")
if torch.cuda.is_available():
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

# Quick test: extract + train + eval on just cable_gland
# Uncomment to run:
# !python -m src.scripts.extract_features
# !python -m src.scripts.train_fusion --category cable_gland --epochs 3
# !python -m src.scripts.evaluate --category cable_gland